# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 clinical case dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

Key features include clinical, hormonal, imaging, and genetic data for three female patients with non-classical 21-hydroxylase deficiency.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}\n")

## 2. Data Overview
Review available record sets and fields using their `@id` identifiers.

In [ ]:
# List all record sets and field information from the dataset
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in metadata. Ensure Croissant schema includes recordSet definitions.")
else:
    for rs in record_sets:
        print(f"Record Set @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', 'N/A')}")
        print(f"  Description: {rs.get('description', 'N/A')}")
        print("  Fields:")
        fields = rs.get('field', [])
        for f in fields:
            print(f"    Field @id: {f['@id']}, Name: {f.get('name', 'N/A')} (Type: {f.get('dataType', 'N/A')})")
        print()
    # Save a list of record set IDs
    record_set_ids = [rs['@id'] for rs in record_sets]

## 3. Data Extraction
Load table data from each record set into a DataFrame. All references use entity `@id` fields for clarity and reproducibility.

In [ ]:
# Load records for each record set using @id reference
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record Set {record_set_id} columns:", df.columns.tolist())
    print(df.head(2), "\n")

# Choose a record set for detailed analysis (use first one found)
selected_record_set_id = record_set_ids[0] if record_set_ids else None
if selected_record_set_id:
    print(f"Selected Record Set: {selected_record_set_id}\nColumn List:", dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps: filter, normalize, and group data. All field references use `@id`. Modify the numeric and group field variables as needed for each record set.

In [ ]:
# Example: Analyze a numeric field from the first record set
rs_df = dataframes[selected_record_set_id]

# Find candidate numeric field by scanning columns
numeric_field_id = None
for col in rs_df.columns:
    if rs_df[col].dtype in ['int64', 'float64'] and rs_df[col].notnull().all():
        numeric_field_id = col
        break

# Example threshold filter and normalization
if numeric_field_id:
    threshold = rs_df[numeric_field_id].mean()  # median or mean as threshold
    filtered_df = rs_df[rs_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Try grouping by a categorical field
    group_field_id = None
    for col in rs_df.columns:
        if rs_df[col].dtype == 'object' and rs_df[col].nunique() <= 3:
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields. All fields are referenced by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Plot histogram of selected numeric field
if numeric_field_id:
    plt.figure(figsize=(6,4))
    sns.histplot(rs_df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    
    # If group_field exists, plot boxplot
    if group_field_id:
        plt.figure(figsize=(6,4))
        sns.boxplot(x=rs_df[group_field_id], y=rs_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading the FAIR^2 dataset, extraction and EDA of clinical, hormonal, imaging, or genetic records using the `mlcroissant` library. All dataset structures and field references utilized their `@id` values for reproducibility.

- Clinical data for three patients were explored.
- Example numeric field (> threshold) filtering, normalization, and grouping carried out.
- Visual previews provided.

Researchers can extend this exploration for deeper insights or integrate with downstream clinical analytics.
